# Conversational AI — Building a Chatbot with OpenAI + Gradio

This notebook builds a working chatbot step by step, starting from the absolute basics
of how `gr.ChatInterface` works, and ending with a streaming, context-aware "clothes store"
sales assistant.

**What you'll learn:**
1. How `gr.ChatInterface` passes `message` and `history` into your callback function.
2. How to turn that history into a proper OpenAI `messages` list.
3. How to stream a response token-by-token instead of waiting for the full reply.
4. How to use the system prompt for **one-shot prompting** — giving the model context,
   tone, and an example of the kind of answer you want.
5. How to dynamically adjust the system prompt based on what the user says.

> **Note:** This notebook is meant to be run cell by cell. Each `gr.ChatInterface(...).launch()`
> call opens a local web UI — try chatting with it, then close the tab (or interrupt the
> kernel) before moving to the next section, since only the *most recently defined* `chat`
> function is used by a new `launch()` call.


## 1. Setup

Import what we need, load the API key from a `.env` file, and initialize the OpenAI client.

Your `.env` file (in the same folder as this notebook) should contain:

```
OPENAI_API_KEY=sk-...your-key...
```


In [16]:
# Imports
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


In [2]:
# Load environment variables from .env
# We only print the key's prefix, never the full key, to avoid leaking secrets
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - please add OPENAI_API_KEY to your .env file")


OpenAI API Key exists and begins sk-proj-


In [3]:
# Initialize the OpenAI client and pick a model
openai_client = OpenAI()
MODEL = 'gpt-4.1-mini'


## 2. How `gr.ChatInterface` works

`gr.ChatInterface` is a specialized version of `gr.Interface` built for chatbots.
It automatically keeps track of the whole conversation for you and passes it into
your callback function on every turn:

```python
def chat(message, history):
    ...
```

- **`message`** — the newest thing the user just typed.
- **`history`** — everything said so far in the conversation, as a list of
  `{"role": ..., "content": ...}` dictionaries (because we pass `type="messages"`
  to `ChatInterface`). This is the same shape the OpenAI API expects for its
  `messages` parameter — that's the whole point.

Gradio manages `history` as UI state: after your function returns a reply, Gradio
appends both the user's message and your response to its internal history, ready
to hand back to you (one entry longer) on the next turn. You never manage it manually.

Below is a minimal example that just proves this — it doesn't call any LLM yet,
it only echoes back what it received.


In [4]:
def chat_demo(message, history):
    return f"You said '{message}' and the history so far is: {history}"

# Try running this, sending a couple of messages, and watch how `history` grows.
# Uncomment to try it interactively:
# gr.ChatInterface(fn=chat_demo, type="messages").launch()


## 3. A real chatbot: forwarding history to the OpenAI API

Now let's replace the placeholder with a proper callback that:

1. Cleans `history` down to just `role` and `content` (Gradio's messages can
   sometimes carry extra metadata fields the OpenAI API doesn't expect).
2. Builds the final `messages` list: **system prompt + history + new message**.
3. Sends it to the OpenAI API and returns the model's reply.

We'll define one global `system_message` variable so it's easy to tweak the
assistant's personality throughout the notebook.


In [5]:
# The assistant's personality / instructions.
# Feel free to edit this and re-run the launch cell below to see the effect.
system_message = "You are a helpful assistant"


In [6]:
def chat(message, history):
    # Keep only the fields the OpenAI API expects
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    # system prompt + everything said so far + the newest user message
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    response = openai_client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [7]:
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


## 4. Streaming the response

Waiting for the entire reply before showing anything feels sluggish for longer
answers. We can stream the response **token by token** instead, so the user sees
words appear as the model generates them — just like ChatGPT's UI.

Two changes make this work:

- `stream=True` in the API call turns the response into an iterator of small
  `chunk` objects instead of one final object.
- Using `yield` instead of `return` turns `chat` into a **generator function**.
  `gr.ChatInterface` understands generator callbacks: each `yield` updates the
  chat bubble in place, rather than waiting for the function to fully finish.

Each chunk contains a small piece of new text in `chunk.choices[0].delta.content`
(which can be `None` for non-text chunks, hence the `or ''` fallback). We keep
building up `response` and re-yield the growing string each time.


In [8]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    stream = openai_client.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response


In [9]:
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


## 5. One-shot prompting: giving the assistant a persona and an example

The `system_message` isn't just "instructions" — it's where you set context, tone,
business rules, and even **example responses** (this is "one-shot prompting": giving
one example of the kind of answer you want).

Here we turn the assistant into a clothes store sales assistant that:
- Gently steers customers toward items on sale.
- Knows hats are 60% off and most other items are 50% off.
- Has an example reply modeled directly in the prompt.

Because `chat()` (defined above, using streaming) reads `system_message` as a
global variable each time it runs, updating `system_message` and re-running the
`launch()` cell is enough to see the new persona in action — no need to redefine
`chat` again.


In [10]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage " \
"the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. " \
"For example, if the customer says 'I'm looking to buy a hat', " \
"you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.' " \
"Encourage the customer to buy hats if they are unsure what to get."


In [11]:
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


### Extending the system prompt

We can keep layering on more business rules using `+=`, without having to
rewrite the whole prompt from scratch.


In [12]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, " \
"but remind the customer to look at hats!"


In [13]:
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


## 6. Making the system prompt dynamic

So far `system_message` has been fixed at the time we call `chat()`. But we can go
further: **inspect the user's message and adjust the system prompt on the fly**,
per turn, before sending it to the model.

Below, if the customer mentions "belt" (a product the store doesn't sell), we
append an extra instruction just for that turn — telling the model to steer the
conversation toward other sale items instead. This extra instruction only affects
`relevant_system_message` for this call; the original `system_message` global is
left untouched, so it doesn't accumulate across turns.


In [14]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    # Start from the base system prompt, and adapt it for this specific message
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."

    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openai_client.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response


In [15]:
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.
